In [ ]:
import sys
from pathlib import Path

# ── Locate project root and register the basis_data package ───────────────────
# Works whether the kernel CWD is notebooks/ or the project root.
for _candidate in [Path.cwd(), Path.cwd().parent]:
    if (_candidate / "basis_data" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from basis_data.pipeline import build_bond_history

bond_history, cds_mappings_filtered = build_bond_history()
bond_history.head(5)

[cache] cds_transform_interpolated_mapped_tickers_2025.parquet is current — loading from disk


cds_transform_interpolated (2716147, 14)


bond_history (raw) (120137, 5)


bond_history (final) (120137, 21)


,security,date,LAST_PRICE,Benchmark_Spread,G_Spread,MATURITY,SECURITY_NAME,target_cds_maturity,ticker,cds_index,...,BOND_ISIN,bbg_cds_ticker,cds_spread,upfront,cds_spread_5y,cds_5y_maturity,cds_spread_5y_fixed,basis_spread,basis_spread_5y_fixed,basis_price
0,BE6364524635 Corp,2026-07-24,98.223,66.923,66.186,2033-05-19,ABIBB 3 3/8 05/19/33,2033-06-20,ABIBB,Main,...,BE6364524635,ABIB CDS EUR SR 5Y D14 Corp,NaN,NaN,NaN,2031-06-20,NaN,NaN,NaN,NaN
1,DE000A4DFLQ6 Corp,2026-07-24,103.818,152.560,149.877,2031-04-01,SHAEFF 5 3/8 04/01/31,2031-06-20,SHAEFF,XO,...,DE000A4DFLQ6,SCHAAG CDS EUR SR 5Y D14 Corp,NaN,NaN,NaN,2031-06-20,NaN,NaN,NaN,NaN
2,FI4000590864 Corp,2026-07-24,96.262,187.537,183.142,2031-05-28,METSA 3 7/8 05/28/31,2031-06-20,METSA,XO,...,FI4000590864,METSA CDS EUR SR 5Y D14 Corp,NaN,NaN,NaN,2031-06-20,NaN,NaN,NaN,NaN
3,FR0011270487 Corp,2026-07-24,84.697,433.888,433.998,2032-06-14,FTI 4 06/14/32,2032-06-20,FTI,XO_Legacy,...,FR0011270487,FTI CDS EUR SR 5Y D14 Corp,NaN,NaN,NaN,2031-06-20,NaN,NaN,NaN,NaN
4,FR0014004R72 Corp,2026-07-24,88.390,82.337,78.790,2030-07-27,ALOFP 0 1/2 07/27/30,2030-12-20,ALOFP,Main,...,FR0014004R72,ALSTOM CDS EUR SR 5Y D14 Corp,NaN,NaN,NaN,2031-06-20,NaN,NaN,NaN,NaN


In [ ]:
import pandas as pd

# Tickers to drop from the summary entirely (e.g. stale/broken proxies),
# applied before the groupby below.
EXCLUDE_TICKERS = ["CSCHLD", "DISH", "X", "UNIT", "ALVGR", "UU PLC"]

# CDS_Index values to drop from the summary entirely (e.g. thin/legacy
# index families not worth tracking), applied before the groupby below.
EXCLUDE_CDS_INDEX = ["Euro_NonIndex", "XO_Legacy", "Main_Legacy"]


def _last_valid(series):
    """Last non-NaN value in a series, or None if all NaN."""
    valid = series.notna()
    return series[valid].iloc[-1] if valid.any() else None


def _basis_window_stats(g, cds_col, basis_col):
    """latest/min/max/mean basis plus the number of obs (days with both
    cds_col and G_Spread available) -- generic helper shared by the
    target-maturity-matched basis and the fixed-5Y-maturity basis."""
    valid = g[cds_col].notna() & g["G_Spread"].notna()
    valid_dates = g.loc[valid, "date"]
    if valid_dates.empty:
        return None, None, None, None, 0
    date_end = valid_dates.iloc[-1]
    latest_basis = g.loc[g["date"] == date_end, basis_col].iloc[-1]
    basis_valid = g.loc[valid, basis_col]
    return latest_basis, basis_valid.min(), basis_valid.max(), basis_valid.mean(), len(basis_valid)


def _summarize_ticker(g):
    """One row of cash-vs-CDS basis summary stats for a single CDS ticker's
    proxy bond. date_start/date_end bound the window where both the bond's
    G-spread and the maturity-matched CDS spread are available, since basis
    is only computable there; latest_CDS/latest_bond_g/obs/basis/min/max/avg/
    range_pct are all computed over that same window (bond G-spread vs. the
    CDS spread at the bond's own target maturity; obs is that window's
    size -- the number of days with both a bond G-spread and CDS spread).
    basis_5y/min_5y/max_5y/avg_5y are the same stats but for the bond's
    G-spread vs. the CDS spread at the fixed 5Y maturity
    (cds_spread_5y_fixed) instead of the bond's own target maturity, since 5Y
    is the reliably liquid point -- these are the primary basis columns (rows
    are exported only when range_5y_pct resolves). avg/avg_5y are the plain
    mean of the basis window, alongside min/max. latest_5y and
    latest_benchmark are informational reference stats (like the equivalent
    chart legend entries), so they're just the last available value in the
    ticker's full history, independent of either basis window. range_pct is
    the latest basis's position within [min, max]: 100% at the historical
    max, 0% at the min."""
    ticker = g.name  # captured before sort_values, which drops the groupby name
    g = g.sort_values("date")
    valid = g["cds_spread"].notna() & g["G_Spread"].notna()
    valid_dates = g.loc[valid, "date"]

    if valid_dates.empty:
        date_start = date_end = pd.NaT
        latest_cds_spread = latest_bond_g = None
    else:
        date_start = valid_dates.iloc[0]
        date_end = valid_dates.iloc[-1]
        last_row = g.loc[g["date"] == date_end].iloc[-1]
        latest_cds_spread = last_row["cds_spread"]
        latest_bond_g = last_row["G_Spread"]

    basis, min_basis, max_basis, avg_basis, obs = _basis_window_stats(g, "cds_spread", "basis_spread")
    range_pct = None
    if basis is not None:
        basis_range = max_basis - min_basis
        range_pct = 100.0 if basis_range == 0 else (basis - min_basis) / basis_range * 100

    basis_5y, min_5y, max_5y, avg_5y, _ = _basis_window_stats(g, "cds_spread_5y_fixed", "basis_spread_5y_fixed")
    range_5y_pct = None
    if basis_5y is not None:
        basis_5y_range = max_5y - min_5y
        range_5y_pct = 100.0 if basis_5y_range == 0 else (basis_5y - min_5y) / basis_5y_range * 100

    latest_5y = _last_valid(g["cds_spread_5y"]) if "cds_spread_5y" in g.columns else None
    latest_benchmark = _last_valid(g["Benchmark_Spread"]) if "Benchmark_Spread" in g.columns else None

    return pd.Series({
        "CDS_Ticker": ticker,
        "CDS_Index": g["cds_index"].iloc[0],
        "bond_name": g["SECURITY_NAME"].iloc[0],
        # grouped/hidden block: identifiers and dates, not needed at a glance
        "bbg_cds_ticker": g["bbg_cds_ticker"].iloc[0],
        "bond_ISIN": g["BOND_ISIN"].iloc[0],
        "date_start": date_start,
        "date_end": date_end,
        "cds_mat": g["target_cds_maturity"].iloc[0],
        "latest_CDS": latest_cds_spread,
        "latest_5y": latest_5y,
        "latest_bond_g": latest_bond_g,
        "latest_benchmark": latest_benchmark,
        "obs": obs,
        "basis": basis,
        "min": min_basis,
        "max": max_basis,
        "avg": avg_basis,
        "range_pct": range_pct,
        "basis_5y": basis_5y,
        "min_5y": min_5y,
        "max_5y": max_5y,
        "avg_5y": avg_5y,
        "range_5y_pct": range_5y_pct,
    })


basis_summary = (
    bond_history[
        ~bond_history["ticker"].isin(EXCLUDE_TICKERS)
        & ~bond_history["cds_index"].isin(EXCLUDE_CDS_INDEX)
    ]
    .groupby("ticker")
    .apply(_summarize_ticker)
    .reset_index(drop=True)
    .sort_values(["CDS_Index", "CDS_Ticker"])
    .reset_index(drop=True)
)
print(basis_summary.shape)
basis_summary.head(10)

(344, 23)


,CDS_Ticker,CDS_Index,bond_name,bbg_cds_ticker,bond_ISIN,date_start,date_end,cds_mat,latest_CDS,latest_5y,...,basis,min,max,avg,range_pct,basis_5y,min_5y,max_5y,avg_5y,range_5y_pct
0,RAKUGRO,Asia,RAKUTN 1.05 12/02/31,RAKUTN CDS JPY SR 5Y D14 Corp,JP396720DMC4,NaT,NaT,2031-12-20,NaN,210.283600,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BAC,Banks,BAC 4.695 04/23/32,BOFA CDS USD SR 5Y D14 Corp,US06051GNA30,2026-04-17,2026-07-23,2032-06-20,64.148399,55.957699,...,-9.995601,-13.040102,0.336200,-6.070527,22.760410,-18.186301,-21.130998,-7.909398,-14.240961,22.271867
2,C,Banks,C 4.846 06/18/32,CINC CDS USD SR 5Y D14 Corp,US17325FBV94,2026-06-12,2026-07-23,2032-06-20,62.423801,53.458599,...,-8.652199,-9.026600,-2.267900,-5.295493,5.539550,-17.617401,-18.137899,-10.801000,-14.248518,7.094250
3,GS,Banks,GS 4.972 06/03/32,GS CDS USD SR 5Y D14 Corp,US38141GF335,2026-05-28,2026-07-23,2032-06-20,63.386101,55.764400,...,-25.112899,-27.817301,-13.320199,-18.297485,18.654773,-32.734600,-35.738802,-21.293000,-25.968057,20.796370
4,JPM,Banks,JPM 4.622 04/23/32,JPMCC CDS USD SR 5Y D14 Corp,US46647PFM32,2026-04-16,2026-07-23,2032-06-20,48.351200,41.085300,...,-26.359800,-28.115500,-14.513599,-20.676559,12.907755,-33.625700,-35.694700,-22.073101,-27.942243,15.189115
5,MS,Banks,MS 4.809 04/16/32,MS CDS USD SR 5Y D14 Corp,US61748UAW27,2026-04-16,2026-07-23,2032-06-20,67.859001,58.413898,...,-15.917999,-24.095297,-8.768101,-14.392003,53.351560,-25.363102,-33.115900,-17.910699,-23.537353,50.987804
6,WFC,Banks,WFC 4.844 05/20/32,WELLFARGO CDS USD SR 5Y D14 Corp,US95000U4J91,2026-05-14,2026-07-23,2032-06-20,58.014702,49.769299,...,-17.122298,-20.571702,-10.422500,-14.578304,33.986943,-25.367701,-28.588399,-18.567502,-22.668542,32.139810
7,APODS,Fin,APODS 6.7 07/29/31,CZ061134 Curncy,US03770DAD57,2026-04-09,2026-07-23,2031-12-20,219.785095,211.113800,...,21.561095,14.680001,64.452590,29.292897,13.825069,12.889800,5.539602,56.929000,18.828854,14.302947
8,ARCC,Fin,ARCC 5.8 03/08/32,CZ061117 Curncy,US04010LBH50,2026-04-13,2026-07-23,2032-06-20,231.535095,209.623001,...,66.894095,49.303896,80.060911,62.817681,57.190853,44.982001,37.058702,71.976607,46.990557,22.691221
9,BCRED,Fin,BCRED 6 01/29/32,CZ061066 Curncy,US09261HBX44,2026-04-09,2026-07-23,2032-06-20,241.989700,220.816696,...,39.020700,22.595007,111.631489,43.552509,18.448273,17.847696,6.452305,99.632313,24.903337,12.229438


In [ ]:
import os

from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.worksheet.table import Table, TableStyleInfo

BASE_OUTPUT_PATH = Path(r"S:\Structured Credit\Matt Stuff\claude_repos\cds_bond_basis\output\bond_cds_basis_summary.xlsx")
BASE_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)


def _output_path_candidates(base_path, max_versions=20):
    """base_path, then base_path with _v1, _v2, ... appended before the
    suffix -- lets the export fall back to a new file instead of erroring
    when the previous export is still open in Excel."""
    yield base_path
    for v in range(1, max_versions + 1):
        yield base_path.with_name(f"{base_path.stem}_v{v}{base_path.suffix}")


# Blank rows reserved above the header. Row 1 is left fully blank (e.g. for
# a title/timestamp added manually in Excel later); row 2 holds the
# live_basis average; row 3 holds all the auto-generated group labels,
# directly above the real header row. Everything derives from HEADER_ROW so
# it stays a single knob to turn.
ROW_OFFSET = 3
HEADER_ROW = ROW_OFFSET + 1
DATA_START_ROW = HEADER_ROW + 1

# Bloomberg field mnemonic for the bond G-spread, kept consistent with the
# field already pulled via bdh_mm() into bond_history's G_Spread column.
G_SPREAD_FIELD = "BLOOMBERG_MID_G_SPREAD"

# live_basis is clamped to this range -- a handful of illiquid/distressed
# names can otherwise print basis values in the thousands that blow out the
# color scale and column width for everything else.
LIVE_BASIS_CAP = 500
# Fixed anchor (not the column's observed min/max, unlike range_pct) for the
# basis/basis_5y diverging scale, centered at 0bp.
BASIS_SCALE_CAP = 200

# Pre-populated manual entries for the cds_override column, keyed by
# CDS_Ticker -- written into the sheet at export time so the manual
# override doesn't need to be re-typed by hand after every refresh.
CDS_OVERRIDES = {"MLTIVR": 350}

FORMULA_FILL = PatternFill(start_color="D9D9D9", end_color="D9D9D9", fill_type="solid")
# Light green flags cds_override as a manual-entry cell, distinct from the
# grey formula-driven live columns around it.
OVERRIDE_FILL = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
# Very light neutral banding for the static columns -- deliberately grey
# rather than a built-in blue/orange Excel table style, so it doesn't
# compete with the red/white/blue diverging scales sitting in the same rows.
BAND_FILL = PatternFill(start_color="F2F2F2", end_color="F2F2F2", fill_type="solid")
# CDS_Index category tint -- pale blue for Main, pale amber for the XO family.
INDEX_FILLS = {
    "Main": PatternFill(start_color="DDEBF7", end_color="DDEBF7", fill_type="solid"),
    "XO": PatternFill(start_color="FCE4D6", end_color="FCE4D6", fill_type="solid"),
}

FONT_NAME = "Calibri"
FONT_SIZE = 8
HEADER_FONT = Font(name=FONT_NAME, size=FONT_SIZE, bold=True)
# Group-label headers (above the real header row) are italicized to read as
# annotations rather than column names.
GROUP_LABEL_FONT = Font(name=FONT_NAME, size=FONT_SIZE, bold=True, italic=True)
BODY_FONT = Font(name=FONT_NAME, size=FONT_SIZE)
CENTER_ALIGN = Alignment(horizontal="center", vertical="center")
# Dark navy for categorical/identifier columns (names, tickers, ISINs) so
# they read as "what this row is" versus the plain-black numeric results.
CATEGORICAL_FONT_COLOR = "1F3864"
# Header row banner: same navy as the categorical font, filled solid with
# bold white text, so the header row reads as a strong divider.
HEADER_ROW_FILL = PatternFill(start_color=CATEGORICAL_FONT_COLOR, end_color=CATEGORICAL_FONT_COLOR, fill_type="solid")
HEADER_ROW_FONT = Font(name=FONT_NAME, size=FONT_SIZE, bold=True, color="FFFFFF")
CATEGORICAL_COLUMNS = {"CDS_Ticker", "CDS_Index", "bond_name", "bbg_cds_ticker", "bond_ISIN"}
BOLD_COLUMNS = {"CDS_Ticker", "bond_name"}
# Thin visual seam between the static (historical Python pull) columns and
# the live (Bloomberg BDP formula) columns.
SECTION_BORDER_SIDE = Side(style="medium", color="808080")
# Light box drawn around the basis-stat column group (see _box_range below).
GROUP_BORDER_SIDE = Side(style="thin", color="B0B0B0")

RANGE_PCT_COLOR_SCALE = dict(
    start_type="num", start_value=0, start_color="F8696B",
    mid_type="num", mid_value=50, mid_color="FFFFFF",
    end_type="num", end_value=100, end_color="5A8AC6",
)
BASIS_COLOR_SCALE = dict(
    start_type="num", start_value=-BASIS_SCALE_CAP, start_color="F8696B",
    mid_type="num", mid_value=0, mid_color="FFFFFF",
    end_type="num", end_value=BASIS_SCALE_CAP, end_color="5A8AC6",
)

# Identifier/date/reference columns, grouped and collapsed since they're rarely needed at a glance.
GROUPED_COLUMNS = [
    "bbg_cds_ticker", "bond_ISIN", "date_start", "date_end", "cds_mat",
    "latest_CDS", "latest_5y", "latest_bond_g", "latest_benchmark",
]
# Separate group: intermediate live lookups feeding spread_live/g_live, not
# usually needed on their own.
LIVE_GROUPED_COLUMNS = {"bond_bench", "bench_price"}


def _iferror(formula, fallback='"--"'):
    """Wrap a '=...' formula string as =IFERROR(..., fallback)."""
    return f"=IFERROR({formula[1:]},{fallback})"


def _box_range(ws, min_row, max_row, min_col, max_col, side):
    """Draw a border box around the perimeter of a cell range, preserving
    each cell's existing border sides that aren't part of the perimeter
    edge being drawn (e.g. the live-block seam set elsewhere)."""
    for row in range(min_row, max_row + 1):
        for col in range(min_col, max_col + 1):
            cell = ws.cell(row=row, column=col)
            existing = cell.border
            cell.border = Border(
                left=side if col == min_col else existing.left,
                right=side if col == max_col else existing.right,
                top=side if row == min_row else existing.top,
                bottom=side if row == max_row else existing.bottom,
            )


COLUMN_WIDTHS = {
    "CDS_Ticker": 12, "bbg_cds_ticker": 26, "CDS_Index": 10, "bond_name": 28, "bond_ISIN": 16,
    "date_start": 12, "date_end": 12, "cds_mat": 12,
    "latest_CDS": 14, "latest_5y": 12, "latest_bond_g": 14, "latest_benchmark": 16,
    "obs": 8, "basis": 8, "min": 8, "max": 8, "avg": 8, "range_pct": 8,
    "basis_5y": 8, "min_5y": 8, "max_5y": 8, "avg_5y": 8, "range_5y_pct": 8,
}
DATE_COLUMNS = {"date_start", "date_end", "cds_mat"}
SPREAD_COLUMNS = {
    "latest_CDS", "latest_5y", "latest_bond_g", "latest_benchmark",
    "basis", "min", "max", "avg",
    "basis_5y", "min_5y", "max_5y", "avg_5y",
}
PERCENT_COLUMNS = {"range_pct", "range_5y_pct"}
INTEGER_COLUMNS = {"obs"}
# basis/basis_5y are the "headline" result columns -- get a diverging scale
# fixed at +/-BASIS_SCALE_CAP centered on 0, on top of the accounting format
# SPREAD_COLUMNS already gives them.
BASIS_SCALE_COLUMNS = {"basis", "basis_5y"}

# Drop tickers with no computable fixed-5Y basis (range_5y_pct is the
# reference column since the 5Y point is the reliably liquid one), rather
# than exporting a row of blanks.
basis_summary_export = basis_summary[basis_summary["range_5y_pct"].notna()].reset_index(drop=True)
print(f"Dropped {len(basis_summary) - len(basis_summary_export)} rows with no basis data")

# ExcelWriter opens the file handle as soon as it's constructed, so a locked
# (currently-open-in-Excel) target raises PermissionError right here -- fall
# back through _v1, _v2, ... until one isn't locked, rather than erroring out.
writer = None
for candidate_path in _output_path_candidates(BASE_OUTPUT_PATH):
    try:
        writer = pd.ExcelWriter(candidate_path, engine="openpyxl")
    except PermissionError:
        continue
    OUTPUT_PATH = candidate_path
    break
if writer is None:
    raise PermissionError("Every candidate output path is locked -- close one of the open workbooks and rerun.")
if OUTPUT_PATH != BASE_OUTPUT_PATH:
    print(f"{BASE_OUTPUT_PATH.name} is open elsewhere -- writing to {OUTPUT_PATH.name} instead")

with writer:
    basis_summary_export.to_excel(writer, sheet_name="Basis Summary", index=False, startrow=ROW_OFFSET)
    ws = writer.sheets["Basis Summary"]

    # Locks the 3 blank rows + header (rows 1-HEADER_ROW) and the first 3 columns.
    ws.freeze_panes = f"D{DATA_START_ROW}"

    last_row = len(basis_summary_export) + HEADER_ROW
    n_static_cols = len(basis_summary_export.columns)

    def _group_label(row, start_col, end_col, text):
        ws.merge_cells(start_row=row, start_column=start_col, end_row=row, end_column=end_col)
        cell = ws.cell(row=row, column=start_col, value=text)
        cell.font = GROUP_LABEL_FONT
        cell.alignment = CENTER_ALIGN

    # All group labels live on the spacer row directly above the real header
    # row, leaving row 1 completely blank. basis_col..range_5y_pct_col cover
    # both the target-maturity-matched and fixed-5Y basis-stat columns.
    group_label_row = HEADER_ROW - 1
    basis_col = basis_summary_export.columns.get_loc("basis") + 1
    range_pct_col = basis_summary_export.columns.get_loc("range_pct") + 1
    basis_5y_col = basis_summary_export.columns.get_loc("basis_5y") + 1
    range_5y_pct_col = basis_summary_export.columns.get_loc("range_5y_pct") + 1

    # Sheet title, above the identifier/obs columns that sit to the left of
    # the "Basis Stats" block (spans the hidden grouped columns in between).
    title_start_col = basis_summary_export.columns.get_loc("CDS_Ticker") + 1
    title_end_col = basis_summary_export.columns.get_loc("obs") + 1
    _group_label(group_label_row, title_start_col, title_end_col, "CDS Bond Basis Historical results")

    # Two labels over the basis-stat block: the target-maturity-matched
    # group (basis/min/max/avg/range_pct), then the fixed-5Y group
    # (basis_5y/min_5y/max_5y/avg_5y/range_5y_pct), boxed together with a light border.
    _group_label(group_label_row, basis_col, range_pct_col, "Maturity-Matched Basis")
    _group_label(group_label_row, basis_5y_col, range_5y_pct_col, "Basis vs 5Y CDS")
    _box_range(ws, group_label_row, last_row, basis_col, range_5y_pct_col, GROUP_BORDER_SIDE)

    for i, col in enumerate(basis_summary_export.columns, start=1):
        col_letter = get_column_letter(i)
        ws.column_dimensions[col_letter].width = COLUMN_WIDTHS.get(col, 14)
        if col in GROUPED_COLUMNS:
            ws.column_dimensions[col_letter].outlineLevel = 1
            ws.column_dimensions[col_letter].hidden = True
        if col in DATE_COLUMNS:
            for cell in ws[col_letter][HEADER_ROW:]:
                cell.number_format = "yyyy-mm-dd"
        elif col in SPREAD_COLUMNS:
            # accounting style: negatives in red parens, matching the slideshow charts
            for cell in ws[col_letter][HEADER_ROW:]:
                cell.number_format = "#,##0.0;[Red](#,##0.0)"
            if col in BASIS_SCALE_COLUMNS:
                ws.conditional_formatting.add(
                    f"{col_letter}{DATA_START_ROW}:{col_letter}{last_row}", ColorScaleRule(**BASIS_COLOR_SCALE)
                )
        elif col in PERCENT_COLUMNS:
            for cell in ws[col_letter][HEADER_ROW:]:
                cell.number_format = "0.0\\%"
            # range_pct is already normalized 0-100, so anchor the color scale to
            # those fixed values (not the observed min/max in the column) --
            # red at 0 (historical low), white at 50, blue at 100 (historical high)
            ws.conditional_formatting.add(
                f"{col_letter}{DATA_START_ROW}:{col_letter}{last_row}", ColorScaleRule(**RANGE_PCT_COLOR_SCALE)
            )
        elif col in INTEGER_COLUMNS:
            for cell in ws[col_letter][HEADER_ROW:]:
                cell.number_format = "#,##0"

    # Neutral grey banding on the static columns only (the live block already
    # has its own uniform grey/green fill, which would look muddy with a
    # second banding pattern layered on top).
    for row in range(DATA_START_ROW + 1, last_row + 1, 2):
        for col_idx in range(1, n_static_cols + 1):
            ws.cell(row=row, column=col_idx).fill = BAND_FILL

    # CDS_Index category tint, overriding the banding fill above for that column.
    cds_index_col_idx = basis_summary_export.columns.get_loc("CDS_Index") + 1
    for row in range(DATA_START_ROW, last_row + 1):
        cell = ws.cell(row=row, column=cds_index_col_idx)
        fill = next((f for key, f in INDEX_FILLS.items() if str(cell.value).startswith(key)), None)
        if fill is not None:
            cell.fill = fill

    # Static columns referenced by the live formulas below.
    isin_col = get_column_letter(basis_summary_export.columns.get_loc("bond_ISIN") + 1)
    cds_ticker_col = get_column_letter(basis_summary_export.columns.get_loc("bbg_cds_ticker") + 1)
    # live_basis is a bond-G-spread-vs-5Y-CDS basis, so its range check uses
    # the fixed-5Y-maturity historical range ("min_5y"/"max_5y"), not the
    # target-maturity one (min/max). dev_mean below reuses avg_5y_col the
    # same way, comparing the live basis against the historical 5Y-basis mean.
    min_basis_col = get_column_letter(basis_summary_export.columns.get_loc("min_5y") + 1)
    max_basis_col = get_column_letter(basis_summary_export.columns.get_loc("max_5y") + 1)
    avg_5y_col_static = get_column_letter(basis_summary_export.columns.get_loc("avg_5y") + 1)

    # Live-refresh columns: re-derive the bond's benchmark, its G-spread, and
    # the CDS-vs-bond basis directly in Excel via the Bloomberg Add-in (BDP),
    # rather than from the historical Python pull. Requires the Add-in to
    # resolve. Each column's formula references the live column(s) to its left.
    # Every formula is wrapped in IFERROR(..., "--") so a missing/unresolved
    # BDP lookup shows as a placeholder instead of an #N/A propagating
    # through the dependent formulas to its right. cds_override is a blank,
    # manually-editable cell (pre-populated from CDS_OVERRIDES for tickers
    # listed there); cds_5y falls back to live_cds_5y whenever it's empty,
    # and live_basis is computed off cds_5y so a manual override feeds
    # straight through to the basis/range calc. dev_mean is the final
    # column, showing how far live_basis sits from the historical 5Y-basis
    # mean (avg_5y, a static column computed in Python).
    live_col_start = n_static_cols + 1
    live_columns = [
        ("bond_live", 8, "#,##0.000"),
        ("bond_bench", 22, None),
        ("bench_price", 14, "#,##0.000"),
        ("spread_live", 8, "#,##0.0;[Red](#,##0.0)"),
        ("g_live", 8, "#,##0.0;[Red](#,##0.0)"),
        ("live_cds_5y", 8, "#,##0.0;[Red](#,##0.0)"),
        ("cds_override", 8, "#,##0.0;[Red](#,##0.0)"),
        ("cds_5y", 8, "#,##0.0;[Red](#,##0.0)"),
        ("live_basis", 8, "#,##0.0;[Red](#,##0.0)"),
        ("live_range_pct", 8, "0.0\\%"),
        ("dev_mean", 8, "#,##0.0;[Red](#,##0.0)"),
    ]
    (bond_live_col, bond_bench_col, bench_price_col, spread_live_col,
     g_live_col, live_cds_5y_col, cds_override_col, cds_5y_col,
     live_basis_col, live_range_pct_col, dev_mean_col) = (
        get_column_letter(live_col_start + offset) for offset in range(len(live_columns))
    )

    for offset, (name, width, _) in enumerate(live_columns):
        col_letter = get_column_letter(live_col_start + offset)
        ws.cell(row=HEADER_ROW, column=live_col_start + offset, value=name).font = HEADER_FONT
        ws.column_dimensions[col_letter].width = width
        if name in LIVE_GROUPED_COLUMNS:
            ws.column_dimensions[col_letter].outlineLevel = 1
            ws.column_dimensions[col_letter].hidden = True

    # Group label above the whole live-refresh block (spans the hidden
    # bond_bench/bench_price columns too, since they sit inside this range).
    _group_label(group_label_row, live_col_start, live_col_start + len(live_columns) - 1, "*Live Basis vs 5y CDS")

    for row in range(DATA_START_ROW, last_row + 1):
        ticker = basis_summary_export.iloc[row - DATA_START_ROW]["CDS_Ticker"]
        ws[f"{bond_live_col}{row}"] = _iferror(f'=BDP({isin_col}{row}&" Corp","LAST_PRICE")')
        ws[f"{bond_bench_col}{row}"] = _iferror(f'=BDP({isin_col}{row}&" Corp","YAS_BENCHMARK_BOND_ISIN")')
        # bond_bench is a bare ISIN (from YAS_BENCHMARK_BOND_ISIN), so it needs
        # the same " CORP" suffix as bond_ISIN to resolve as a BDP security.
        ws[f"{bench_price_col}{row}"] = _iferror(f'=BDP({bond_bench_col}{row}&" CORP","LAST_PRICE")')
        ws[f"{spread_live_col}{row}"] = _iferror(
            f'=BDP({isin_col}{row}&" Corp","YAS_YLD_SPREAD",'
            f'"YAS_BOND_PX="&{bond_live_col}{row},"YAS_BNCHMRK_BOND_PX="&{bench_price_col}{row})'
        )
        ws[f"{g_live_col}{row}"] = _iferror(
            f'=BDP({isin_col}{row}&" Corp","{G_SPREAD_FIELD}",'
            f'"YAS_BOND_PX="&{bond_live_col}{row},"YAS_BNCHMRK_BOND_PX="&{bench_price_col}{row})'
        )
        ws[f"{live_cds_5y_col}{row}"] = _iferror(f'=BDP({cds_ticker_col}{row},"LAST_PRICE")')
        # cds_override left blank, unless the ticker has a pre-populated
        # value in CDS_OVERRIDES -- manual entry otherwise, no formula.
        if ticker in CDS_OVERRIDES:
            ws[f"{cds_override_col}{row}"] = CDS_OVERRIDES[ticker]
        ws[f"{cds_5y_col}{row}"] = _iferror(
            f'=IF(ISBLANK({cds_override_col}{row}),{live_cds_5y_col}{row},{cds_override_col}{row})'
        )
        # clamped to [-LIVE_BASIS_CAP, +LIVE_BASIS_CAP] so a handful of
        # illiquid/distressed names can't blow out the column's scale/color rule
        ws[f"{live_basis_col}{row}"] = _iferror(
            f'=MAX(-{LIVE_BASIS_CAP},MIN({LIVE_BASIS_CAP},{cds_5y_col}{row}-{g_live_col}{row}))'
        )
        ws[f"{live_range_pct_col}{row}"] = _iferror(
            f'=({live_basis_col}{row}-{min_basis_col}{row})'
            f'/({max_basis_col}{row}-{min_basis_col}{row})*100'
        )
        ws[f"{dev_mean_col}{row}"] = _iferror(f'={live_basis_col}{row}-{avg_5y_col_static}{row}')

    for offset, (name, width, number_format) in enumerate(live_columns):
        col_letter = get_column_letter(live_col_start + offset)
        # light grey flags formula-driven (live BDP pull) columns; cds_override
        # gets the light green "manual entry" fill instead.
        fill = OVERRIDE_FILL if name == "cds_override" else FORMULA_FILL
        for cell in ws[col_letter][HEADER_ROW:]:
            cell.fill = fill
            if number_format is not None:
                cell.number_format = number_format

    # Same fixed 0/50/100 red-white-blue scale as range_pct, applied to its live counterpart.
    ws.conditional_formatting.add(
        f"{live_range_pct_col}{DATA_START_ROW}:{live_range_pct_col}{last_row}", ColorScaleRule(**RANGE_PCT_COLOR_SCALE)
    )

    # Column-average summary, 2 rows above the header -- lands in one of the
    # blank spacer rows, directly above the live_basis column.
    avg_row = HEADER_ROW - 2
    avg_cell = ws[f"{live_basis_col}{avg_row}"]
    avg_cell.value = f"=AVERAGE({live_basis_col}{DATA_START_ROW}:{live_basis_col}{last_row})"
    avg_cell.number_format = "#,##0.0;[Red](#,##0.0)"
    avg_cell.font = HEADER_FONT

    # Column indices (not letters) for the categorical/bold font treatment below.
    categorical_col_idxs = {
        basis_summary_export.columns.get_loc(c) + 1 for c in CATEGORICAL_COLUMNS
    }
    bold_col_idxs = {basis_summary_export.columns.get_loc(c) + 1 for c in BOLD_COLUMNS}

    # Calibri 8pt + centered everywhere -- applied before the table is added
    # so the table style doesn't need to fight with per-cell formatting.
    # Categorical/identifier columns get a dark navy font (bold for
    # CDS_Ticker/bond_name) so they read as "what this row is" versus the
    # plain-black numeric result columns.
    last_col_letter = get_column_letter(live_col_start + len(live_columns) - 1)
    for row in ws.iter_rows(min_row=HEADER_ROW, max_row=last_row, min_col=1, max_col=live_col_start + len(live_columns) - 1):
        for cell in row:
            is_header = cell.row == HEADER_ROW
            is_categorical = (not is_header) and cell.column in categorical_col_idxs
            is_bold = is_header or ((not is_header) and cell.column in bold_col_idxs)
            cell.font = Font(
                name=FONT_NAME, size=FONT_SIZE, bold=is_bold,
                color=CATEGORICAL_FONT_COLOR if is_categorical else None,
            )
            cell.alignment = CENTER_ALIGN

    # Header row banner: solid navy fill + bold white text across every
    # column (static and live), overriding the black/bold set just above.
    header_last_col = live_col_start + len(live_columns) - 1
    for col in range(1, header_last_col + 1):
        cell = ws.cell(row=HEADER_ROW, column=col)
        cell.fill = HEADER_ROW_FILL
        cell.font = HEADER_ROW_FONT

    # Thin seam between the static (historical) block and the live
    # (Bloomberg BDP formula) block.
    for row in range(HEADER_ROW, last_row + 1):
        cell = ws.cell(row=row, column=live_col_start)
        cell.border = Border(left=SECTION_BORDER_SIDE)

    # Real Excel Table (not just autofilter) spanning every column, static +
    # live -- gives sortable/filterable headers and a named range other
    # formulas/pivots can reference. Row striping is handled manually above
    # (BAND_FILL) rather than via the table style, so it stays neutral grey
    # instead of Excel's built-in blue/orange bands.
    table = Table(displayName="BasisSummary", ref=f"A{HEADER_ROW}:{last_col_letter}{last_row}")
    table.tableStyleInfo = TableStyleInfo(
        name="TableStyleLight1",
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=False,
        showColumnStripes=False,
    )
    ws.add_table(table)

print(f"Wrote {len(basis_summary_export)} rows to {OUTPUT_PATH}")

Dropped 2 rows with no basis data


Wrote 342 rows to S:\Structured Credit\Matt Stuff\claude_repos\cds_bond_basis\output\bond_cds_basis_summary.xlsx


In [ ]:
import xlwings as xw
import pywintypes

# ThisWorkbook VBA: forces manual calculation on open (this workbook is
# formula-heavy with live BDP lookups, so auto-calc on open/edit is slow)
# and exposes a manual/automatic toggle. Injected via the Excel COM API
# (xlwings) since openpyxl can't write VBA -- requires "Trust access to the
# VBA project object model" enabled in Excel's Trust Center > Macro Settings.
VBA_THISWORKBOOK_CODE = """Private Sub Workbook_Open()
    Application.Calculation = xlCalculationManual
    Application.CalculateBeforeSave = False
    ' MsgBox "Yea Nice"
End Sub


Sub ToggleCalculationMode()
    With Application
        If .Calculation = xlCalculationManual Then
            .Calculation = xlCalculationAutomatic
            'MsgBox "Calculation mode set to Automatic", vbInformation
        Else
            .Calculation = xlCalculationManual
            'MsgBox "Calculation mode set to Manual", vbInformation
        End If
    End With
End Sub
"""

FINAL_OUTPUT_PATH = None
app = xw.App(visible=False, add_book=False)
try:
    wb = app.books.open(str(OUTPUT_PATH))
    try:
        code_module = wb.api.VBProject.VBComponents("ThisWorkbook").CodeModule
    except pywintypes.com_error as exc:
        raise RuntimeError(
            "Excel blocked VBA access -- enable 'Trust access to the VBA project "
            "object model' in File > Options > Trust Center > Trust Center Settings "
            "> Macro Settings, then rerun this cell."
        ) from exc
    if code_module.CountOfLines > 0:
        code_module.DeleteLines(1, code_module.CountOfLines)
    code_module.AddFromString(VBA_THISWORKBOOK_CODE)

    # Same locked-file fallback as the .xlsx export above, but for the
    # macro-enabled .xlsm final artifact (e.g. a previous run's .xlsm still
    # open in Excel via the os.startfile call at the bottom of this cell).
    for xlsm_candidate in _output_path_candidates(BASE_OUTPUT_PATH.with_suffix(".xlsm")):
        try:
            wb.api.SaveAs(str(xlsm_candidate), FileFormat=52)  # xlOpenXMLWorkbookMacroEnabled
        except pywintypes.com_error:
            continue
        FINAL_OUTPUT_PATH = xlsm_candidate
        break
    if FINAL_OUTPUT_PATH is None:
        raise PermissionError("Every candidate .xlsm output path is locked -- close one of the open workbooks and rerun.")
    wb.close()
finally:
    app.quit()

OUTPUT_PATH.unlink()  # superseded by the macro-enabled .xlsm
print(f"Added VBA to ThisWorkbook -- macro-enabled workbook saved to {FINAL_OUTPUT_PATH}")
os.startfile(FINAL_OUTPUT_PATH)

Added VBA to ThisWorkbook -- macro-enabled workbook saved to S:\Structured Credit\Matt Stuff\claude_repos\cds_bond_basis\output\bond_cds_basis_summary.xlsm
